# 03 分段插值与三次样条

分段方法不再强迫一个高次多项式穿过所有数据点，而是在每个小区间上构造局部近似。这通常能提高稳健性，也使单个节点变化的影响更加局部。


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS",
    "PingFang SC",
    "Heiti SC",
    "SimHei",
    "Noto Sans CJK SC",
    "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False

ROOT = pathlib.Path.cwd()
while not (ROOT / "src" / "py_sc").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from py_sc import NaturalCubicSpline, lagrange_interpolate, piecewise_linear_interpolate


## 分段线性插值

在区间 $[x_i,x_{i+1}]$ 上，分段线性插值为

$$
p(x)=\frac{x_{i+1}-x}{x_{i+1}-x_i}y_i+
\frac{x-x_i}{x_{i+1}-x_i}y_{i+1}.
$$

它简单、局部、稳健，但在节点处导数通常不连续。


In [ ]:
x = np.array([0.0, 0.8, 1.7, 2.5, 3.2, 4.0])
y = np.cos(1.5 * x) + 0.2 * x
x_eval = np.linspace(x.min(), x.max(), 500)

linear = piecewise_linear_interpolate(x, y, x_eval)
polynomial = lagrange_interpolate(x, y, x_eval)

plt.figure(figsize=(8, 4.5))
plt.plot(x_eval, polynomial, label="全局多项式插值")
plt.plot(x_eval, linear, label="分段线性插值")
plt.scatter(x, y, color="black", zorder=3, label="数据点")
plt.xlabel("x")
plt.ylabel("函数值")
plt.title("全局插值与局部插值的对比")
plt.legend()
plt.tight_layout()
plt.show()


## 局部性实验

改变一个数据点会影响全局多项式的整个曲线；对分段线性插值，这种影响只出现在相邻区间。


In [ ]:
y_perturbed = y.copy()
y_perturbed[3] += 0.8

poly_perturbed = lagrange_interpolate(x, y_perturbed, x_eval)
linear_perturbed = piecewise_linear_interpolate(x, y_perturbed, x_eval)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
axes[0].plot(x_eval, polynomial, label="原始数据")
axes[0].plot(x_eval, poly_perturbed, label="扰动后")
axes[0].scatter(x, y_perturbed, color="black", s=20)
axes[0].set_title("全局多项式")
axes[0].set_xlabel("x")
axes[0].set_ylabel("函数值")
axes[0].legend()

axes[1].plot(x_eval, linear, label="原始数据")
axes[1].plot(x_eval, linear_perturbed, label="扰动后")
axes[1].scatter(x, y_perturbed, color="black", s=20)
axes[1].set_title("分段线性")
axes[1].set_xlabel("x")
axes[1].legend()

fig.suptitle("局部方法限制了单个数据变化的影响范围")
fig.tight_layout()
plt.show()


## 自然三次样条

三次样条插值（cubic spline interpolation）在每个小区间上使用

$$
S_i(x)=a_i+b_i(x-x_i)+c_i(x-x_i)^2+d_i(x-x_i)^3,
$$

并要求 $S$、$S'$ 和 $S''$ 在内部节点处连续。自然边界条件为

$$
S''(x_0)=S''(x_n)=0.
$$

由此得到的线性系统是三对角系统，可以高效求解。


In [ ]:
spline = NaturalCubicSpline.fit(x, y)
spline_values = spline(x_eval)

plt.figure(figsize=(8, 4.5))
plt.plot(x_eval, linear, label="分段线性插值")
plt.plot(x_eval, spline_values, label="自然三次样条")
plt.scatter(x, y, color="black", zorder=3, label="数据点")
plt.xlabel("x")
plt.ylabel("函数值")
plt.title("样条插值引入更高阶的光滑性")
plt.legend()
plt.tight_layout()
plt.show()

print("各区间三次多项式系数")
for i, (a, b, c, d) in enumerate(zip(spline.a, spline.b, spline.c, spline.d)):
    print(f"区间 {i}: a={a: .4f}, b={b: .4f}, c={c: .4f}, d={d: .4f}")


## 边界条件扩展路线

当前实现使用自然边界条件。后续可以加入固定导数边界条件，也就是给定端点一阶导数的 clamped spline，并比较端点行为如何变化。


## 小结

* 分段线性插值稳健、局部，但不光滑。
* 自然三次样条在保持局部性的同时，使一阶和二阶导数连续。
* 样条求解会导出三对角线性系统，这也为后续线性方程组章节做铺垫。
